In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Extended LamaH-CE (BiasCast, HESS 2026) - tải dữ liệu và kết quả thí nghiệm của tác giả
# Bài: Konold và cs., "BiasCast: learning and adjusting real time biases from meteorological forecasts to enhance runoff predictions", HESS 30:5067–5096 (2026). Dữ liệu Zenodo 17119635 (CC BY-NC 4.0), kết quả thí nghiệm Zenodo 17292895.
#
# MỤC LỤC
#   Phần 1 - Thư viện, hằng số, hàm tải theo luồng dùng chung
#   Phần 2 - Tải Extended LamaH-CE (0,95 GB) và kết quả thí nghiệm của tác giả (0,23 GB) → 2 file ZIP

In [2]:
# Phần 1 — Thư viện, hằng số, hàm tải theo luồng dùng chung
# Đọc file .tar.gz từ Zenodo theo luồng (không lưu file nén gốc) và ghi thẳng mọi file vào MỘT file ZIP
# (tránh giới hạn số file của output Kaggle; pandas đọc thẳng CSV trong ZIP được)
# Log mỗi 60 giây. Thiếu dòng "Kiểm tra ZIP" ở cuối log = tải lỗi, chạy lại cell
import io
import shutil
import sys
import tarfile
import time
import urllib.request
import zipfile
from pathlib import Path

SOURCES = {  # tên ZIP lưu ra: (URL Zenodo, dung lượng nén dự phòng khi Zenodo không trả Content-Length, GB)
    "lamah_ce_ext.zip": ("https://zenodo.org/records/17119635/files/Extended_LamaH-CE_daily.tar.gz?download=1", 0.95),
    "biascast_experiments.zip": ("https://zenodo.org/records/17292895/files/Experiments.tar.gz?download=1", 0.23),
}
ZIP_LEVEL = 6
LOG_EVERY_SECONDS = 60
READ_BUFFER_BYTES = 8 * 1024 * 1024
NETWORK_TIMEOUT_S = 300             # mạng treo quá 5 phút thì báo lỗi thay vì đứng im

IS_KAGGLE = Path("/kaggle/working").exists()
IS_COLAB = "google.colab" in sys.modules
BASE_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("/content") if IS_COLAB else Path.cwd()


class CountingReader(io.RawIOBase):
    # Bọc luồng HTTP để đếm số byte nén đã đọc (theo dõi tiến độ)

    def __init__(self, raw):
        self.raw, self.bytes_read = raw, 0

    def readable(self):
        return True

    def readinto(self, buffer):
        chunk = self.raw.read(len(buffer))
        n = len(chunk)
        buffer[:n] = chunk
        self.bytes_read += n
        return n


def stream_tar_to_zip(url: str, fallback_gb: float, zip_path: Path) -> None:
    # Tải .tar.gz theo luồng, ghi mọi file vào zip_path (giữ nguyên đường dẫn gốc), in tiến độ và kiểm tra CRC
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"\n{zip_path.name}")
    print(f"Nền tảng: {'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Local'} | "
          f"ổ đĩa trống: {shutil.disk_usage(zip_path.parent).free / 1e9:.1f} GB")
    response = urllib.request.urlopen(url, timeout=NETWORK_TIMEOUT_S)
    total_bytes = int(response.headers.get("Content-Length") or fallback_gb * 1e9)
    counter = CountingReader(response)
    stream = io.BufferedReader(counter, buffer_size=READ_BUFFER_BYTES)

    t0 = last_log = time.time()
    kept = raw_bytes = 0
    groups = {}
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=ZIP_LEVEL,
                         allowZip64=True) as zf, tarfile.open(fileobj=stream, mode="r|gz") as archive:
        for member in archive:
            if not member.isfile():
                continue
            member_file = archive.extractfile(member)
            zf.writestr(member.name, member_file.read() if member_file else b"")
            kept += 1
            raw_bytes += member.size
            key = "/".join(member.name.split("/")[:4][:-1]) or member.name
            groups[key] = groups.get(key, 0) + 1
            if time.time() - last_log > LOG_EVERY_SECONDS:
                last_log = time.time()
                print(f"[{(last_log - t0) / 60:5.1f} phút] {counter.bytes_read / total_bytes:6.1%} "
                      f"| {kept} file | đang ở: {'/'.join(member.name.split('/')[:3])}", flush=True)

    print(f"XONG sau {(time.time() - t0) / 60:.1f} phút | đọc {counter.bytes_read / 1e9:.2f} GB nén | "
          f"{kept} file | gốc {raw_bytes / 1e9:.2f} GB → ZIP {zip_path.stat().st_size / 1e9:.2f} GB")
    for key in sorted(groups):
        print(f"  {key}: {groups[key]} file")
    with zipfile.ZipFile(zip_path) as zf:   # kiểm tra ZIP đọc lại được, không hỏng
        print(f"Kiểm tra ZIP: {len(zf.namelist())} mục, lỗi CRC: {zf.testzip()}")


In [3]:
# Phần 2 — Tải Extended LamaH-CE (0,95 GB) và kết quả thí nghiệm của tác giả (0,23 GB) => 2 file ZIP
# lamah_ce_ext.zip: toàn bộ dữ liệu ngày (A_basins_total_upstrm, D_gauges có qmin/qmean/qmax)
# biascast_experiments.zip: 24 cấu hình thí nghiệm (config.yml, best_model.pt, scaler, test_metrics.csv)
for zip_name, (url, fallback_gb) in SOURCES.items():
    stream_tar_to_zip(url, fallback_gb, BASE_DIR / zip_name)


lamah_ce_ext.zip
Nền tảng: Kaggle | ổ đĩa trống: 20.9 GB
[  1.0 phút]  13.3% | 125 file | đang ở: LamaH_extended/A_basins_total_upstrm/2_timeseries
[  2.0 phút]  26.6% | 249 file | đang ở: LamaH_extended/A_basins_total_upstrm/2_timeseries
[  3.0 phút]  39.8% | 375 file | đang ở: LamaH_extended/A_basins_total_upstrm/2_timeseries
[  4.0 phút]  52.2% | 493 file | đang ở: LamaH_extended/A_basins_total_upstrm/2_timeseries
[  5.0 phút]  65.5% | 627 file | đang ở: LamaH_extended/A_basins_total_upstrm/2_timeseries
[  6.1 phút]  78.8% | 746 file | đang ở: LamaH_extended/A_basins_total_upstrm/2_timeseries
[  7.1 phút]  89.4% | 848 file | đang ở: LamaH_extended/A_basins_total_upstrm/2_timeseries
[  8.1 phút] 100.0% | 1876 file | đang ở: LamaH_extended/D_gauges/2_timeseries
XONG sau 8.1 phút | đọc 0.95 GB nén | 1991 file | gốc 2.81 GB → ZIP 0.95 GB
  LamaH_extended/A_basins_total_upstrm/1_attributes: 3 file
  LamaH_extended/A_basins_total_upstrm/2_timeseries: 859 file
  LamaH_extended/A_basins_to